In [ ]:

# =============================================================================
# Multi-seed robustness: 10 seeds
#   ConvNeXt-Large (unmasked)  vs  Swin-Tiny (masked, ground-truth/oracle)
# Widens the seed pool from 3 to 10 to answer the "only three seeds" objection.
# NOTE: torch is pinned BEFORE it is first imported, because Kaggle's default
# build dropped the sm_60 kernels that the Tesla P100 needs.
# =============================================================================
import subprocess, sys, os, json, time, random, glob

gpu = ""
try:
    gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                         capture_output=True, text=True, timeout=60).stdout.strip()
except Exception as e:
    print("nvidia-smi failed:", e)
print("GPU reported by nvidia-smi:", gpu or "(none)")

if "P100" in gpu:
    print("P100 detected -> installing the cu121 torch build (sm_60 support)", flush=True)
    subprocess.run([sys.executable,"-m","pip","install","-q",
                    "torch==2.5.1","torchvision==0.20.1",
                    "--index-url","https://download.pytorch.org/whl/cu121"], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q","timm==1.0.11"], check=False)

import numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", flush=True)
if torch.cuda.is_available():
    print("compute capability:", torch.cuda.get_device_capability(0))
    torch.zeros(8, device="cuda")            # fail fast if the build is wrong
    print("GPU smoke test OK", flush=True)

# ---- locate the dataset ----------------------------------------------------
root4 = None
for d, subs, _ in os.walk("/kaggle/input"):
    if {"unmasked","masked"} <= set(subs): root4 = d; break
assert root4, "could not find the unmasked/masked condition folders"
print("dataset root ->", root4)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [0,1,2,3,4,5,6,7,8,9]
EPOCHS, BS, LR, IMG = 12, 16, 1e-4, 224

tf_train = transforms.Compose([
    transforms.Resize((IMG,IMG)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2), transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_eval = transforms.Compose([
    transforms.Resize((IMG,IMG)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def loaders(cond, seed):
    tr = datasets.ImageFolder(f"{root4}/{cond}/train", tf_train)
    va = datasets.ImageFolder(f"{root4}/{cond}/valid", tf_eval)
    te = datasets.ImageFolder(f"{root4}/{cond}/test",  tf_eval)
    g = torch.Generator(); g.manual_seed(seed)
    return (DataLoader(tr,batch_size=BS,shuffle=True,num_workers=2,generator=g),
            DataLoader(va,batch_size=BS,num_workers=2),
            DataLoader(te,batch_size=BS,num_workers=2), tr)

def class_weights(ds):
    c = np.bincount([y for _,y in ds.samples], minlength=3).astype(float)
    return torch.tensor(c.sum()/(3*np.maximum(c,1)), dtype=torch.float32, device=DEV)

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); P=[]; Y=[]
    for x,y in dl:
        P += list(model(x.to(DEV)).argmax(1).cpu().numpy()); Y += list(y.numpy())
    P=np.array(P); Y=np.array(Y)
    return (float(100*(P==Y).mean()),
            float(100*np.mean([(P[Y==c]==c).mean() for c in np.unique(Y)])),
            P.tolist(), Y.tolist())

def train_one(arch, cond, seed):
    seed_all(seed)
    trl, val, tel, trds = loaders(cond, seed)
    model = timm.create_model(arch, pretrained=True, num_classes=3).to(DEV)
    opt  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss(weight=class_weights(trds))
    use_amp = (DEV=="cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_va, best_state = -1, None
    for ep in range(EPOCHS):
        model.train()
        for x,y in trl:
            x,y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        va,_,_,_ = evaluate(model, val)
        if va > best_va:
            best_va = va
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(f"    {arch[:16]:<16} {cond:<8} seed{seed} ep{ep+1:02d} val={va:.2f}", flush=True)
    model.load_state_dict(best_state)
    acc,bal,P,Y = evaluate(model, tel)
    del model; torch.cuda.empty_cache()
    return dict(arch=arch,cond=cond,seed=seed,best_val=best_va,
                test_acc=acc,test_bal=bal,preds=P,labels=Y)

RUNS = [("convnext_large","unmasked"), ("swin_tiny_patch4_window7_224","masked")]
OUT  = "/kaggle/working/multiseed10_results.json"
out, t0 = [], time.time()
for seed in SEEDS:
    for arch,cond in RUNS:
        try:
            r = train_one(arch,cond,seed); out.append(r)
            print(f"  >> {arch[:20]:<20} {cond:<8} seed{seed}: TEST {r['test_acc']:.2f} "
                  f"bal {r['test_bal']:.2f}  [{(time.time()-t0)/60:.1f} min]", flush=True)
        except Exception as e:
            print(f"  !! {arch} {cond} seed{seed} FAILED: {type(e).__name__}: {e}", flush=True)
        json.dump(out, open(OUT,"w"), indent=1)     # checkpoint after every run

import statistics as st
print("\n=== SUMMARY ===")
summary={}
for arch,cond in RUNS:
    a=[r["test_acc"] for r in out if r["arch"]==arch and r["cond"]==cond]
    b=[r["test_bal"] for r in out if r["arch"]==arch and r["cond"]==cond]
    if not a: continue
    summary[f"{arch}|{cond}"]={"n":len(a),"mean":st.mean(a),
        "sd_sample":st.stdev(a) if len(a)>1 else 0.0,"accs":a,
        "bal_mean":st.mean(b),"bal_sd_sample":st.stdev(b) if len(b)>1 else 0.0}
    print(f"  {arch[:22]:<22} {cond:<8} n={len(a)} mean={st.mean(a):.2f} "
          f"sd(n-1)={st.stdev(a) if len(a)>1 else 0:.2f}")
json.dump({"summary":summary,"runs":out,"seeds":SEEDS,"epochs":EPOCHS,"gpu":gpu},
          open(OUT,"w"), indent=1)
print("saved", OUT)
